In [1]:

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------
# Project paths and setup
# -----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
#PROJECT_ROOT = Path(__file__).resolve().parents[1]
PREP_ROOT = PROJECT_ROOT / "data" / "prepared"
FEATURE_ROOT = PROJECT_ROOT / "data" / "featured" / "v1"

INTERACTIONS_PATH = PREP_ROOT / "interactions_prepared.csv"
PRODUCTS_PATH = PREP_ROOT / "products_prepared.csv"

USER_FEATURES_PATH = FEATURE_ROOT / "user_features.parquet"
ITEM_FEATURES_PATH = FEATURE_ROOT / "item_features.parquet"
INTERACTION_FEATURES_PATH = FEATURE_ROOT / "interaction_features.parquet"


def min_max_scale(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=series.index)

    min_val = s.min()
    max_val = s.max()

    if pd.isna(min_val) or pd.isna(max_val):
        return pd.Series(np.nan, index=series.index)

    if min_val == max_val:
        return pd.Series(0.0, index=series.index)

    return (s - min_val) / (max_val - min_val)


def ensure_required_columns(df: pd.DataFrame, required: list[str], df_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")


def load_prepared_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not INTERACTIONS_PATH.exists():
        raise FileNotFoundError(f"Missing file: {INTERACTIONS_PATH}")

    if not PRODUCTS_PATH.exists():
        raise FileNotFoundError(f"Missing file: {PRODUCTS_PATH}")

    interactions = pd.read_csv(INTERACTIONS_PATH)
    products = pd.read_csv(PRODUCTS_PATH)

    ensure_required_columns(
        interactions,
        ["user_id", "item_id", "event", "event_weight", "event_ts", "event_date"],
        "interactions_prepared.csv",
    )
    ensure_required_columns(
        products,
        ["category"],
        "products_prepared.csv",
    )

    interactions["user_id"] = pd.to_numeric(interactions["user_id"], errors="coerce").astype("Int64")
    interactions["item_id"] = pd.to_numeric(interactions["item_id"], errors="coerce").astype("Int64")
    interactions["event"] = interactions["event"].astype(str).str.strip().str.lower()
    interactions["event_weight"] = pd.to_numeric(interactions["event_weight"], errors="coerce")
    interactions["event_ts"] = pd.to_datetime(interactions["event_ts"], errors="coerce")
    interactions["event_date"] = pd.to_datetime(interactions["event_date"], errors="coerce")

    interactions = interactions.dropna(subset=["user_id", "item_id", "event", "event_weight", "event_ts"])
    interactions["user_id"] = interactions["user_id"].astype("int64")
    interactions["item_id"] = interactions["item_id"].astype("int64")

    products.columns = [c.strip() for c in products.columns]

    return interactions, products


def normalize_products(products: pd.DataFrame) -> pd.DataFrame:
    products = products.copy()

    if "item_id" not in products.columns:
        if "product_id" in products.columns:
            products["item_id"] = pd.to_numeric(products["product_id"], errors="coerce")
        elif "id" in products.columns:
            products["item_id"] = pd.to_numeric(products["id"], errors="coerce")

    if "item_id" not in products.columns:
        raise ValueError("products_prepared.csv must contain one of: item_id, product_id, or id")

    products["item_id"] = pd.to_numeric(products["item_id"], errors="coerce")
    products = products.dropna(subset=["item_id"]).copy()
    products["item_id"] = products["item_id"].astype("int64")

    if "category" in products.columns:
        products["category"] = (
            products["category"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        products["category"] = "unknown"

    if "price_norm" not in products.columns:
        products["price_norm"] = min_max_scale(products["price"]) if "price" in products.columns else 0.0

    if "rating_norm" not in products.columns:
        if "rating" in products.columns:
            products["rating_norm"] = min_max_scale(products["rating"])
        else:
            products["rating_norm"] = 0.0

    if "stock_norm" not in products.columns:
        if "stock" in products.columns:
            products["stock_norm"] = min_max_scale(products["stock"])
        else:
            products["stock_norm"] = 0.0

    keep_cols = ["item_id", "price_norm", "rating_norm", "stock_norm", "category"]
    return products[keep_cols].drop_duplicates(subset=["item_id"]).reset_index(drop=True)


def build_user_features(interactions: pd.DataFrame) -> pd.DataFrame:
    max_ts = interactions["event_ts"].max()

    user_features = (
        interactions.groupby("user_id")
        .agg(
            user_total_interactions=("user_id", "size"),
            user_view_count=("event", lambda s: (s == "view").sum()),
            user_addtocart_count=("event", lambda s: (s == "addtocart").sum()),
            user_transaction_count=("event", lambda s: (s == "transaction").sum()),
            user_avg_event_weight=("event_weight", "mean"),
            user_active_days=("event_date", lambda s: s.dt.date.nunique()),
            last_event_ts=("event_ts", "max"),
        )
        .reset_index()
    )

    user_features["user_last_seen_recency_days"] = (
        (max_ts - user_features["last_event_ts"]).dt.total_seconds() / 86400.0
    )

    user_features = user_features.drop(columns=["last_event_ts"])

    return user_features


def build_item_features(interactions: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    item_features = (
        interactions.groupby("item_id")
        .agg(
            item_total_interactions=("item_id", "size"),
            item_view_count=("event", lambda s: (s == "view").sum()),
            item_addtocart_count=("event", lambda s: (s == "addtocart").sum()),
            item_transaction_count=("event", lambda s: (s == "transaction").sum()),
            item_avg_event_weight=("event_weight", "mean"),
        )
        .reset_index()
    )

    item_features["item_popularity_rank"] = (
        item_features["item_total_interactions"]
        .rank(method="dense", ascending=False)
        .astype("int64")
    )

    products_norm = normalize_products(products)

    item_features = item_features.merge(products_norm, on="item_id", how="left")

    item_features["category"] = item_features["category"].fillna("unknown")
    for col in ["price_norm", "rating_norm", "stock_norm"]:
        item_features[col] = item_features[col].fillna(0.0)

    return item_features


def build_interaction_features(interactions: pd.DataFrame) -> pd.DataFrame:
    max_ts = interactions["event_ts"].max()

    interaction_features = (
        interactions.groupby(["user_id", "item_id"])
        .agg(
            user_item_event_weight_sum=("event_weight", "sum"),
            user_item_interaction_count=("item_id", "size"),
            last_event_ts=("event_ts", "max"),
        )
        .reset_index()
    )

    interaction_features["user_item_last_event_recency_days"] = (
        (max_ts - interaction_features["last_event_ts"]).dt.total_seconds() / 86400.0
    )

    interaction_features = interaction_features.drop(columns=["last_event_ts"])

    return interaction_features


def save_feature_sets(
    user_features: pd.DataFrame,
    item_features: pd.DataFrame,
    interaction_features: pd.DataFrame,
) -> None:
    FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

    user_features.to_parquet(USER_FEATURES_PATH, index=False)
    item_features.to_parquet(ITEM_FEATURES_PATH, index=False)
    interaction_features.to_parquet(INTERACTION_FEATURES_PATH, index=False)

    print(f"Saved: {USER_FEATURES_PATH}")
    print(f"Saved: {ITEM_FEATURES_PATH}")
    print(f"Saved: {INTERACTION_FEATURES_PATH}")


def main() -> None:
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Loading: {INTERACTIONS_PATH}")
    print(f"Loading: {PRODUCTS_PATH}")

    interactions, products = load_prepared_data()

    user_features = build_user_features(interactions)
    item_features = build_item_features(interactions, products)
    interaction_features = build_interaction_features(interactions)

    save_feature_sets(user_features, item_features, interaction_features)

    print("\nShapes:")
    print(f"user_features: {user_features.shape}")
    print(f"item_features: {item_features.shape}")
    print(f"interaction_features: {interaction_features.shape}")

    print("\nSample user_features:")
    print(user_features.head().to_string(index=False))

    print("\nSample item_features:")
    print(item_features.head().to_string(index=False))

    print("\nSample interaction_features:")
    print(interaction_features.head().to_string(index=False))


if __name__ == "__main__":
    main()


Project root: C:\Users\barath\recomart-pipeline
Loading: C:\Users\barath\recomart-pipeline\data\prepared\interactions_prepared.csv
Loading: C:\Users\barath\recomart-pipeline\data\prepared\products_prepared.csv


Saved: C:\Users\barath\recomart-pipeline\data\featured\v1\user_features.parquet
Saved: C:\Users\barath\recomart-pipeline\data\featured\v1\item_features.parquet
Saved: C:\Users\barath\recomart-pipeline\data\featured\v1\interaction_features.parquet

Shapes:
user_features: (1407580, 8)
item_features: (235061, 11)
interaction_features: (2145179, 5)

Sample user_features:
 user_id  user_total_interactions  user_view_count  user_addtocart_count  user_transaction_count  user_avg_event_weight  user_active_days  user_last_seen_recency_days
       0                        3                3                     0                       0                    1.0                 1                     6.253132
       1                        1                1                     0                       0                    1.0                 1                    35.384506
       2                        8                8                     0                       0                    1.0          